# Overview

This example uploads the data NLR delivers for one physical piece: the folder becomes a Sample Set with one Sample per measured pad, and two runs over those same pads — an XRF map and a DC I-V sweep — each a Measurement Set with one Measurement per Sample and its Setup, the delivered tables as files, and one Property per measured quantity.
A delivered folder holds the XRF grid table, the DC I-V tables and photographs of the piece, and re-running the notebook adds only what is missing.

## Install the API client

The samples, measurements and files endpoints are not released yet, so the client is installed from its branch until it merges. Restart the kernel after this cell.

In [ ]:
%pip install -r requirements.txt

## Set Parameters

- **HOST**: platform the data is uploaded to
- **DATA_DIR**: the delivered folder beside this notebook — the XRF grid table, the DC I-V tables and the photographs
- **PHYSICAL_ID**: the identifier written on the physical piece the measured pads are part of — every Sample carries it
- **XRF_INSTRUMENT**: the machine the XRF map was measured on
- **IV_INSTRUMENT**: the machine the DC I-V sweep was measured on
- **ACCOUNT_SLUG**: account the data belongs to, empty for the default account
- **FILES**: which files to upload

In [ ]:
import urllib.parse

HOST = "https://alphafilm.mat3ra.com"
DATA_DIR = "nlr"
PHYSICAL_ID = ""
XRF_INSTRUMENT = ""
IV_INSTRUMENT = ""
ACCOUNT_SLUG = ""
FILES = ["records"]  # the delivered tables and photographs; [] uploads none

url = urllib.parse.urlsplit(HOST)
address = {
    "host": url.hostname,
    "port": url.port or (443 if url.scheme == "https" else 80),
    "secure": url.scheme == "https",
}

## Authenticate and initialize API client

### Authenticate
Authenticate in the browser (OIDC device flow) or via JupyterLite host injection. Credentials are stored in environment variables.

### Initialize API client
Create an authenticated API client and resolve the owner account ID.

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate(**address)

# Imports

In [ ]:
from pathlib import Path

from parse_nlr import parse_nlr
from run_document import load, serialize
from upload_run import account_id, upload

## Parse the delivered folder

Read the folder into the documents the platform stores: one Sample Set, and one run per technique over its Samples. Nothing is uploaded yet.

In [ ]:
# reading the delivery and writing one run document per technique: nothing here talks to the platform
runs = []
for parsed in parse_nlr(Path(DATA_DIR), PHYSICAL_ID, XRF_INSTRUMENT, IV_INSTRUMENT):
    document_path = serialize(parsed, "parsed", name=parsed["run"].replace(" ", "_") + ".json")
    run = load(document_path)
    runs.append(run)
    print(
        f"{run['physicalId']}: {len(run['samples'])} samples (ordered set) · run {run['run']}: "
        f"{len(run['measurements'])} measurements (ordered set, one per sample) · "
        f"{len(run['set_files'])} files · {len(run['properties'])} properties · {document_path}"
    )

## Select the account

`ACCOUNT_SLUG` re-authenticates the client against that account, so the run is read and written there.

In [ ]:
if ACCOUNT_SLUG:
    client = APIClient.authenticate(account_id=account_id(client, ACCOUNT_SLUG), **address)

## Upload the data

Create the Sample Set and its Samples, then a Measurement Set per technique with one Measurement per Sample, the delivered files and the Properties.

In [ ]:
for run in runs:
    upload(client, run, files=FILES)

## Find the data in the web app

Each technique is a folder in the account's Measurements tab.

In [ ]:
print(f"Open {HOST}, your account's Measurements tab: {', '.join(run['run'] for run in runs)}")

## References

- [Mat3ra REST API](https://docs.mat3ra.com/rest-api/overview/)